In [ ]:
import pandas as pd
from dataclasses import dataclass
import os
import matplotlib.pyplot as plt

In [ ]:
# Note: 'slip_dataset_all.pkl' require pandas version 2.1.4 (python 3.10)
dataset = 'tests/rosbag_test_data/marmotte/ga_hard_snow_25_01_a/slip_dataset_all.pkl'
root = os.path.join(os.getcwd(), '../../..')
data_path = os.path.join(root, dataset)

dataset_snow = pd.read_pickle(data_path)

# ToDo: add a small dataset to the repository for example purpose instead

### Requirement:
The dataframe must contain one trajectory or a batch of trajectories (one per row) with some features (column) containing a timestep index in there name eg: `feature_1`, `feature_2` ...

In [ ]:
dataset_snow.head()

# Using *trajectory_dataclass_tools*

```python
def aggregate_multiple_features_from_dataframe(
        dataset_frame: pd.DataFrame,
        dataset_info: str,
        features_config: dict[str, Union[Type[AbstractFeatureDataclass], tuple[str, ...]]]
    ) -> dataclass:
    ...
```

- param `dataset_frame`: Dataset as a panda dataframe
- param `dataset_info`: Any relevant information pertaining to the dataset (location, robot, condition ...)
- param `features_config`: The features to agregate from the dataset as a configuration dictionary

## 1. Base case

Extract multiple features from a dataset (formated in a dataframe) based on a configuration dictionary.

The `features_config` specify the feature name to lookout in the `dataset_frame` header and agregate them in a `Multifeature` dataclass. Feature dimensions such as 'x', 'y' 'z' are specified either by using existing `AbstractFeatureDataclass` subclass such as: `StatePose2D`, `CmdStandard`, `CmdSkidSteer`, `Velocity`, `VelocitySkidSteer` or by using tuple of strings such as `('<new feature dataclass type name>', '<dimension names 1>', '<dimension names 2>', ...)`.

        >>> feature_config = {
        >>>             'icp_interpolated': StatePose2D,
        >>>             'idd_vel':          StatePose2D,
        >>>             'icp':              ('StatePose3D', 'x', 'y', 'z', 'roll', 'pitch', 'yaw')
        >>>             }

Note that each `AbstractFeatureDataclass` subclass validate that each dimension have uniform shape and have monotonic increassing timestep index without skip


In [ ]:
from trajectory_container_tools.dataframe_tools import aggregate_multiple_features_from_dataframe
from trajectory_container_tools.dataclasses.panda_dataframe_feature_dataclass import StatePose2D

In [ ]:
features_config_1 = {
    'body_vel_disturption': StatePose2D,
    'icp_interpolated': StatePose2D,
    'icp_vel': StatePose2D,
    'idd_vel': StatePose2D,
    'icp':     ('StatePose3D', 'x', 'y', 'z', 'roll', 'pitch', 'yaw')
}

mf1 = aggregate_multiple_features_from_dataframe(dataset_snow,
                                  dataset_info="Robot: marmotte, Details: ga_hard_snow_25_01_a",
                                  features_config=features_config_1)

In [ ]:
print(mf1)


## 2. Case requiring post-processing

1. Just create a new dataclass inheriting from a `AbstractFeatureDataclass` subclass e.g. `StatePose2D`
2. Overide `post_init_feature_callback` with the desired post-processing process

In [ ]:
steady_state_mask = dataset_snow['steady_state_mask'].to_numpy() == True

@dataclass
class StatePose2DSteadyState(StatePose2D):
    def post_init_feature_callback(self, feature_name):
        feature_name = self.__getattribute__(feature_name)
        self.__setattr__(feature_name, feature_name[steady_state_mask])
        return None

In [ ]:
features_config_steady_state = {
    'body_vel_disturption': StatePose2DSteadyState,
    'icp_interpolated': StatePose2DSteadyState,
    'icp_vel': StatePose2DSteadyState,
    'idd_vel': StatePose2DSteadyState,
}

mfs = aggregate_multiple_features_from_dataframe(dataset_snow,
                                  dataset_info="Robot: marmotte, Details: ga_hard_snow_25_01_a, STEADY STATE",
                                  features_config=features_config_steady_state)

In [ ]:
print(mfs)

## You can access each features and their dimension by property call

In [ ]:
mf1.idd_vel.x.shape == mf1.icp_vel.x.shape == mf1.body_vel_disturption.x.shape

## and print the multifeature of any feature summary

In [ ]:
print(mf1.body_vel_disturption)

aggregate_multiple_features_from_dataframe

In [ ]:
def plot_dataset_trajectory_steady_state_x_array(trajectory_id: int, title: str, y_label:str, ylim: tuple=(-2.5, 2.5)):
    fig = plt.figure(num=None, figsize=(7,3), dpi=None, facecolor=None, edgecolor=None)
    plt.title(f'{title}   (TRJ ID {trajectory_id})')
    plt.plot(mfs.idd_vel.x[trajectory_id, :], label="steady_state_idd_body_vel_x", color="green")
    plt.plot(mfs.icp_vel.x[trajectory_id, :], label="steady_state_icp_body_vel_x", color="blue", linestyle="dotted")
    plt.plot(mfs.body_vel_disturption.x[trajectory_id, :], label="steady_state_body_vel_disturption_x", color="red", linestyle="dashed", linewidth="2.")
    plt.legend()
    plt.ylabel(y_label)
    plt.xlabel('Timestep')
    plt.ylim(*ylim)
    return None

# _ids = [62,29,30,31,32]
_ids = range(23,25)
for _trj_id in _ids:
    plot_dataset_trajectory_steady_state_x_array(trajectory_id=_trj_id, title="dataset_snow › steady_state", y_label="x", ylim=(-2.5, 2.5))

In [ ]:
def plot_dataset_trajectory_steady_state_yaw(trajectory_id: int, title: str, y_label:str, ylim: tuple=(-2.5, 2.5)):
    fig = plt.figure(num=None, figsize=(7,3), dpi=None, facecolor=None, edgecolor=None)
    plt.title(f'{title}   (TRJ ID {trajectory_id})')
    plt.plot(mfs.idd_vel.yaw[trajectory_id, :], label="steady_state_idd_body_vel_yaw", color="green")
    plt.plot(mfs.icp_vel.yaw[trajectory_id, :], label="steady_state_icp_body_vel_yaw", color="blue", linestyle="dotted")
    plt.plot(mfs.body_vel_disturption.yaw[trajectory_id, :], label="steady_state_body_vel_disturption_yaw", color="red", linestyle="dashed", linewidth="2.")
    plt.legend()
    plt.ylabel(y_label)
    plt.xlabel('Timestep')
    plt.ylim(*ylim)
    return None

# _ids = [29,30,31,32]
_ids = range(23,25)
for _trj_id in _ids:
    plot_dataset_trajectory_steady_state_yaw(trajectory_id=_trj_id, title="dataset_snow › steady_state", y_label="yaw", ylim=(-3.1416, 3.1416))